In [1]:
import json
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.metrics import brier_score_loss
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_json('/workspace/Qwen_2.5_1.5B_judge_answer.json')

In [8]:
FULL = [
    "p_internal",
    "p_consistency",
    "p_semantic",
    "p_selfeval"
]

ABLATIONS = {
    # Single features
    "TokenProb": [
        "p_internal"
    ],

    "SelfConsistency": [
        "p_consistency"
    ],

    "SemanticSimilarity": [
        "p_semantic"
    ],

    "SelfEval": [
        "p_selfeval"
    ],

    # Full model
    "FullFusion": FULL,

    # Leave-one-out
    "-TokenProb": [
        "p_consistency",
        "p_semantic",
        "p_selfeval"
    ],

    "-SelfConsistency": [
        "p_internal",
        "p_semantic",
        "p_selfeval"
    ],

    "-SemanticSimilarity": [
        "p_internal",
        "p_consistency",
        "p_selfeval"
    ],

    "-SelfEval": [
        "p_internal",
        "p_consistency",
        "p_semantic"
    ]
}

In [9]:
def compute_ece(
    confidences,
    labels,
    n_bins=10
):

    bins = np.linspace(
        0,
        1,
        n_bins + 1
    )

    ece = 0

    for i in range(n_bins):

        mask = (
            (confidences >= bins[i]) &
            (confidences < bins[i+1])
        )

        if mask.sum() == 0:
            continue

        acc = labels[mask].mean()

        conf = confidences[mask].mean()

        ece += (
            mask.mean()
            * abs(acc - conf)
        )

    return ece

In [10]:
def evaluate_features(df, features):

    X = df[features].replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.mean())

    y = df["correctness"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = LogisticRegression(
        max_iter=2000,
        C=0.1,

        class_weight="balanced"
    )

    model.fit(X_train, y_train)

    conf = model.predict_proba(X_test)[:, 1]

    return {
        "AUROC": roc_auc_score(y_test, conf),
        "ECE": compute_ece(conf, y_test.to_numpy()),
        "Brier": brier_score_loss(y_test, conf)
    }

In [11]:
results = []

for name, features in ABLATIONS.items():

    metrics = evaluate_features(
        df,
        features
    )

    results.append({

        "Method": name,

        "Features": ", ".join(features),

        "AUROC":
            metrics["AUROC"],

        "ECE":
            metrics["ECE"],

        "Brier":
            metrics["Brier"]
    })

In [12]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "AUROC",
    ascending=False
)

print(results_df)

                Method                                           Features  \
4           FullFusion  p_internal, p_consistency, p_semantic, p_selfeval   
7  -SemanticSimilarity              p_internal, p_consistency, p_selfeval   
6     -SelfConsistency                 p_internal, p_semantic, p_selfeval   
5           -TokenProb              p_consistency, p_semantic, p_selfeval   
1      SelfConsistency                                      p_consistency   
2   SemanticSimilarity                                         p_semantic   
8            -SelfEval              p_internal, p_consistency, p_semantic   
3             SelfEval                                         p_selfeval   
0            TokenProb                                         p_internal   

      AUROC       ECE     Brier  
4  0.676508  0.409202  0.234116  
7  0.676277  0.409508  0.234003  
6  0.676277  0.409508  0.234003  
5  0.672702  0.410005  0.235129  
1  0.640180  0.417429  0.238716  
2  0.640180  0.417429  0.

In [13]:
full_auroc = results_df[
    results_df.Method == "FullFusion"
]["AUROC"].iloc[0]

In [14]:
drops = []

for feature in [
    "-TokenProb",
    "-SelfConsistency",
    "-SemanticSimilarity",
    "-SelfEval"
]:

    score = results_df[
        results_df.Method == feature
    ]["AUROC"].iloc[0]

    drops.append({
        "Removed Feature": feature,
        "AUROC Drop":
            full_auroc - score
    })

drops_df = pd.DataFrame(drops)

print(
    drops_df.sort_values(
        "AUROC Drop",
        ascending=False
    )
)

       Removed Feature  AUROC Drop
3            -SelfEval    0.037135
0           -TokenProb    0.003806
1     -SelfConsistency    0.000231
2  -SemanticSimilarity    0.000231


In [15]:
df.corr(numeric_only=True)["correctness"]

correctness      1.000000
p_internal       0.053202
p_consistency    0.108968
p_semantic       0.108968
p_selfeval       0.076107
Name: correctness, dtype: float64

P_internal = (token prediction) important feature related to correct answer, and other signals shows noisy and unrelated